# load_openaire_researchproduct_sources

Prototipo del nodo `load_openaire_researchproduct_sources` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_sources(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_sources = df.loc[:,['id','sources', *_EXTRACTED_META_COLS]]
    df_research_sources.dropna(inplace=True)
    
    df_research_sources = df_research_sources.explode('sources').reset_index(drop=True)

    df_research_sources = _add_openaire_loaded_metadata(df_research_sources)

    return df_research_sources


In [ ]:
df_research_sources = load_openaire_researchproduct_sources(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_research_sources', 'rows': len(df_research_sources), 'columns': len(df_research_sources.columns)}])


In [ ]:
df_research_sources.head(2)
